# Building a Chatbot with RAG from Scratch 🤖📚
### A Hands-On Guide to Retrieval-Augmented Generation, Agents, and Hallucination Prevention

---

### What we'll build today

1. A **local LLM chatbot** (no cloud API)
2. A **document pipeline**: load → chunk → embed → store
3. A **simple RAG chain** with anti-hallucination prompting
4. An **Agentic RAG** system using CrewAI (Retriever + Verifier)
5. A **Gradio chat UI** you can share with a public link
6. A **head-to-head comparison**: Plain LLM vs Simple RAG vs Agentic RAG

### How to run this

- **Runtime → Run all** (or just execute cells top-to-bottom)
- GPU is **not required** but helpful. To enable: *Runtime → Change runtime type → T4 GPU*
- Setup (Section 0) takes ~2-3 minutes — start it first, then read ahead

## Section 0 — Setup

We install Ollama (a runtime for running open-source LLMs locally), pull two small models,
install Python dependencies, and download *Frankenstein*.

🕐 **Timing:** ~2-3 minutes total. Kick this off and wait.

In [20]:
# Sanity check
import sys, subprocess
assert sys.version_info >= (3, 10), "Python 3.10+ required"
try:
    gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    print("✅ GPU available" if gpu.returncode == 0 else "⚠️  No GPU — running on CPU (fine, just slower)")
except Exception:
    print("⚠️  No GPU — running on CPU (fine, just slower)")
print(f"✅ Python {sys.version.split()[0]}")

✅ GPU available
✅ Python 3.12.13


In [21]:
# Install Ollama (the LLM runtime). ~30 seconds.
!apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3

>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [22]:
# Start the Ollama server as a background process
import os, subprocess, time, requests

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait for the server to be ready
for i in range(20):
    try:
        if requests.get('http://127.0.0.1:11434', timeout=1).ok:
            print(f"✅ Ollama server running (took {i+1}s)")
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️  Ollama server did not start — try running this cell again")

✅ Ollama server running (took 2s)


In [23]:
# Pull the models. llama3.2:3b is a solid small model (~2GB) — good quality, fast on CPU.
# If you're on a constrained runtime, swap for llama3.2:1b (smaller, faster, less accurate).
print("Pulling llama3.2:3b (~2GB)...")
!ollama pull llama3.2:3b

print("\nPulling nomic-embed-text (~270MB)...")
!ollama pull nomic-embed-text

print("\n✅ Models ready:")
!ollama list

Pulling llama3.2:3b (~2GB)...


Pulling nomic-embed-text (~270MB)...


✅ Models ready:
NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    Less than a second ago    


In [24]:
# Install Python dependencies
# (Versions pinned to known-working combinations as of the workshop date)
%pip install -q --upgrade \
    "pandas==2.2.2" \
    "requests==2.32.4" \
    "opentelemetry-api<1.39" \
    "opentelemetry-sdk<1.39" \
    langchain \
    langchain-core \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    langchain-text-splitters \
    langgraph \
    chromadb \
    crewai \
    gradio \
    pypdf

print("✅ Dependencies installed")

import os
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"  # disables OpenTelemetry tracing globally

✅ Dependencies installed


In [25]:
# Download Frankenstein from Project Gutenberg
import urllib.request

url = "https://www.gutenberg.org/files/84/84-0.txt"
urllib.request.urlretrieve(url, "frankenstein.txt")

with open("frankenstein.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"✅ Downloaded {len(text):,} characters")
print(f"\nPreview (skipping Gutenberg header):")
print(text[1400:1800])

✅ Downloaded 419,434 characters

Preview (skipping Gutenberg header):
light. There, Margaret, the sun is for ever
visible, its broad disk just skirting the horizon and diffusing a
perpetual splendour. There—for with your leave, my sister, I will put
some trust in preceding navigators—there snow and frost are banished;
and, sailing over a calm sea, we may be wafted to a land surpassing in
wonders and in beauty every region hitherto discovered on the habitable
globe. 


---

## Section 1 — Talk to the LLM (no RAG yet)

Before we build anything fancy, let's see what the raw LLM knows about Frankenstein.

🔮 **Predict first:** what will a small 3B-parameter model say is the **name of the monster**?

In [26]:
import ollama

def ask_plain_llm(question, model='llama3.2:3b'):
    '''Ask the LLM with no context, no retrieval — just the question.'''
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': question}],
        options={'temperature': 0},
    )
    return response['message']['content']

# The classic trick question
q = "What is the name of the monster in Mary Shelley's Frankenstein?"
print(f"Q: {q}\n")
print(f"A: {ask_plain_llm(q)}")

Q: What is the name of the monster in Mary Shelley's Frankenstein?

A: The creature, also known as "the monster," in Mary Shelley's novel "Frankenstein" (1818) is not given a specific name. It is often referred to as "the creature," "the being," or simply "it." The creature is the product of Victor Frankenstein's experiments and is brought to life through a spark of electricity.

However, it's worth noting that Mary Shelley does refer to the creature as "Adam" in some versions of the novel. This name was likely chosen because Adam is the first man created by God in the biblical account of creation.

It's also worth mentioning that Victor Frankenstein himself refers to the creature as "the demon" or "the fiend," which reflects his fear and revulsion towards it.


💡 **Observation:** In the novel, **the monster has no name.** Small LLMs often confidently say "Frankenstein" — that's the name of the *scientist*, not the creature.

This is **hallucination**: plausible-sounding text that isn't true. It happens because the LLM pattern-matches popular culture rather than the actual book.

**Fix:** give the model the real text to read. That's RAG.

---

## Section 2 — Load and chunk the book

We can't just stuff the whole book into a prompt (LLMs have context limits, and it's wasteful).
Instead, we split it into ~800-character chunks with some overlap, so we can retrieve only the relevant pieces later.

**Why these numbers?**
- `chunk_size=800` — big enough for ~2 paragraphs, small enough for precise retrieval
- `chunk_overlap=100` — prevents key sentences from being cut in half at chunk boundaries
- `separators=["\n\n", "\n", ". ", " "]` — prefer to split at paragraph → sentence → word boundaries

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Load the book
with open("frankenstein.txt", "r", encoding="utf-8") as f:
    raw = f.read()

# Strip Project Gutenberg's legal header/footer
start_marker = "Letter 1"
end_marker = "*** END OF THE PROJECT GUTENBERG"
start = raw.find(start_marker)
end = raw.find(end_marker)
book = raw[start:end] if start > 0 and end > 0 else raw

# Split into overlapping chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = [Document(page_content=c) for c in splitter.split_text(book)]

print(f"📖 Book: {len(book):,} characters → {len(chunks)} chunks")
print(f"\n--- Chunk #50 (sample) ---")
print(chunks[50].page_content[:500])

📖 Book: 419,243 characters → 748 chunks

--- Chunk #50 (sample) ---
He then told me that he would commence his narrative the next day when I
should be at leisure. This promise drew from me the warmest thanks. I have
resolved every night, when I am not imperatively occupied by my duties, to
record, as nearly as possible in his own words, what he has related during
the day. If I should be engaged, I will at least make notes. This
manuscript will doubtless afford you the greatest pleasure; but to me, who
know him, and who hear it from his own lips—with what interes


---

## Section 3 — Embed and store in a vector database

<p align="center">
  <img src="https://raw.githubusercontent.com/hakanotal/workshop-rag-chatbot/main/figures/indexing-phase.png" alt="Indexing phase" width="560"/>
</p>

Now we turn each chunk into a **vector** (a list of numbers that captures its meaning) using `nomic-embed-text`.
Similar meanings → similar vectors → we can find relevant chunks by similarity search.

We store everything in **ChromaDB**, a lightweight local vector database.

🕐 **Timing:** ~30-60 seconds to embed all chunks on CPU.

In [28]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
import shutil, os

# Clean slate (useful if re-running)
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Build the vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    # persist_directory="./chroma_db",
)

print(f"✅ Indexed {len(chunks)} chunks into ChromaDB")

✅ Indexed 748 chunks into ChromaDB


In [29]:
# Test: find passages semantically related to a question
# (note: we haven't called the LLM yet — this is pure retrieval)
query = "Does the monster have a name?"
results = vectorstore.similarity_search(query, k=3)

for i, r in enumerate(results, 1):
    print(f"--- Match {i} ---")
    print(r.page_content[:300])
    print()

--- Match 1 ---
His tale is connected and told with an appearance of the simplest truth,
yet I own to you that the letters of Felix and Safie, which he showed me,
and the apparition of the monster seen from our ship, brought to me a
greater conviction of the truth of his narrative than his asseverations,
however ea

--- Match 2 ---
yellow light of the moon, as it forced its way through the window
shutters, I beheld the wretch—the miserable monster whom I had
created. He held up the curtain of the bed; and his eyes, if eyes they
may be called, were fixed on me. His jaws opened, and he muttered some
inarticulate sounds, while a 

--- Match 3 ---
them; sometimes it was the watery, clouded eyes of the monster, as I
first saw them in my chamber at Ingolstadt.



💡 **Notice:** retrieval already "finds" passages that discuss how the creature is referred to in the book — without any LLM involvement. Retrieval and generation are separate concerns.

---

## Section 4 — Simple RAG chain

<p align="center">
  <img src="https://raw.githubusercontent.com/hakanotal/workshop-rag-chatbot/main/figures/rag-diagram.jpg" alt="Naive RAG pipeline" width="320"/>
  <img src="https://raw.githubusercontent.com/hakanotal/workshop-rag-chatbot/main/figures/retrieval-phase.png" alt="Retrieval phase detail" width="560"/>
</p>

Time to wire it together:

```
question → retrieve top-k chunks → stuff into prompt → LLM → grounded answer
```

The **prompt** is our most important hallucination defense. We explicitly tell the model:
- Use **only** the provided context
- Say *"The text does not specify"* if the answer isn't there
- Do **not** use outside knowledge

This single instruction does more for accuracy than most fancy techniques.

> 📚 The indexing + retrieval diagrams above are adapted from Xuan-Son Nguyen's excellent [*Code a simple RAG from scratch*](https://huggingface.co/blog/ngxson/make-your-own-rag) blog post.

In [77]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model="llama3.2:3b", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

RAG_PROMPT = ChatPromptTemplate.from_template('''You are a literary assistant answering questions about the novel Frankenstein by Mary Shelley.

Use ONLY the context below to answer. If the answer is not in the context, reply exactly:
"The text does not specify."
Do not use any outside knowledge. Do not speculate.

Context:
{context}

Question: {question}

Answer:''')

def format_docs(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

simple_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ Simple RAG chain built")

✅ Simple RAG chain built


In [78]:
# The payoff — same question as Section 1, now grounded in the actual book
q = "What is the name of the monster in Mary Shelley's Frankenstein?"
print(f"Q: {q}\n")
print(f"A: {simple_rag_chain.invoke(q)}")

Q: What is the name of the monster in Mary Shelley's Frankenstein?

A: The text does not specify.


🎯 **That's RAG.** The LLM didn't get smarter — we just gave it the right text to read.

Let's try a few more:

In [80]:
for question in [
    "Who rescues Victor from the Arctic ice?",
    "What books does the creature find and read in the forest?",
    "What is the capital of Switzerland?",  # ← not in the book — should refuse
]:
    print(f"Q: {question}")
    print(f"A: {simple_rag_chain.invoke(question)}\n")

Q: Who rescues Victor from the Arctic ice?
A: The text does not specify.

Q: What books does the creature find and read in the forest?
A: The creature finds a leathern portmanteau containing several articles of dress and some books, including _Paradise Lost_, a volume of _Plutarch's Lives_, and the _Sorrows of Werter_.

Q: What is the capital of Switzerland?
A: The text does not specify.



💡 **The "capital of Switzerland" example is important.** A plain LLM would happily answer "Bern" — a correct fact, but **out of scope**. Good RAG systems know when to stay in their lane.

---

## Section 5 — Agentic RAG with CrewAI

Simple RAG has a weakness: it trusts the first retrieval. If the top-k chunks miss key context, the answer suffers — and there's no double-check.

**Agentic RAG** introduces multiple LLM-powered agents that collaborate. Ours has two:

1. **🔍 Retriever Agent** — searches the book, drafts an answer with quotes
2. **✅ Verifier Agent** — reviews the draft, flags unsupported claims, produces the final answer

The verifier is the key insight. It acts like an editor demanding citations — it rejects anything not directly supported by the retrieved passages.

In [84]:
from crewai import Agent, Task, Crew, Process, LLM

crew_llm = LLM(
    model="ollama/llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0,
    tracing=False,
)

answerer_agent = Agent(
    role="Literary Researcher",
    goal=("Draft a grounded answer using ONLY the passages provided in the task, "
          "quoting the text directly when possible."),
    backstory=("You are a scholar of 19th century gothic literature. "
               "You ground every claim in the actual text and quote passages verbatim."),
    llm=crew_llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3,
)

verifier_agent = Agent(
    role="Fact Verifier",
    goal=("Cross-check every claim in the draft against the passages provided. "
          "Keep claims that are supported; correct or remove claims that are not."),
    backstory=("You are a skeptical editor who demands textual evidence. "
               "You only say 'The text does not specify.' when the passages truly "
               "do not contain the answer — never as a default when unsure."),
    llm=crew_llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3,
)

print("✅ Agents defined")

✅ Agents defined


In [86]:
def run_agentic_rag(question: str, k: int = 10) -> str:
    '''Run the two-agent RAG pipeline for one question.'''

    # 1) Deterministic retrieval — identical to Simple RAG.
    docs = vectorstore.similarity_search(question, k=k)
    passages = "\n\n---\n\n".join(d.page_content for d in docs)

    # 2) Answerer drafts a grounded answer from the passages.
    draft_task = Task(
        description=(
            f"Question: {question}\n\n"
            f"Passages from Frankenstein:\n{passages}\n\n"
            "Write a concise answer that quotes the passages above where useful. "
            "If — and only if — the passages do not contain the answer, reply exactly: "
            "\"The text does not specify.\""
        ),
        expected_output="A draft answer grounded in the provided passages, with direct quotes when possible.",
        agent=answerer_agent,
    )

    # 3) Verifier sees BOTH the passages AND the draft — so it can actually
    #    verify, rather than defaulting to refusal when the draft is thin.
    verify_task = Task(
        description=(
            f"Question: {question}\n\n"
            f"Passages from Frankenstein:\n{passages}\n\n"
            "Review the draft answer from the Literary Researcher.\n"
            "- KEEP every claim that is supported by the passages above.\n"
            "- CORRECT or remove claims that are NOT supported.\n"
            "- Do NOT introduce facts that aren't in the passages.\n"
            "- Only reply 'The text does not specify.' when the passages truly "
            "contain no answer to the question.\n\n"
            "Output only the final verified answer — no preamble, no meta-commentary."
        ),
        expected_output="The final verified answer, grounded only in the provided passages.",
        agent=verifier_agent,
        context=[draft_task],
    )

    crew = Crew(
        agents=[answerer_agent, verifier_agent],
        tasks=[draft_task, verify_task],
        process=Process.sequential,
        verbose=False,
    )

    return str(crew.kickoff())


q = "Who is the captain of the ship that rescues Victor?"
print(f"Q: {q}\n")
print(f"A: {run_agentic_rag(q)}")

Q: Who is the captain of the ship that rescues Victor?



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Literary Researcher                                                                                     │
│                                                                                                                 │
│  Task: Question: Who is the captain of the ship that rescues Victor?                                            │
│                                                                                                                 │
│  Passages from Frankenstein:                                                                                    │
│  In the morning, however, as soon as it was light, I went upon deck and                                         │
│  found all the sailors busy on one side of the vessel, apparently                                               │
│  talking to someone in the sea. It was, in fact, a sledge, like that we                                         │
│  had seen before, which had drifted towards us in the night on a large                                          │
│  fragment of ice. Only one dog remained alive; but there was a human                                            │
│  being within it whom the sailors were persuading to enter the vessel.                                          │
│  He was not, as the other traveller seemed to be, a savage inhabitant of                                        │
│  some undiscovered island, but a European. When I appeared on deck the                                          │
│  master said, “Here is our captain, and he will not allow you to perish                                         │
│  on the open sea.”                                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  The master is a person of an excellent disposition and is remarkable in the                                    │
│  ship for his gentleness and the mildness of his discipline. This                                               │
│  circumstance, added to his well-known integrity and dauntless courage, made                                    │
│  me very desirous to engage him. A youth passed in solitude, my best years                                      │
│  spent under your gentle and feminine fosterage, has so refined the                                             │
│  groundwork of my character that I cannot overcome an intense distaste to                                       │
│  the usual brutality exercised on board ship: I have never believed it to be                                    │
│  necessary, and when I heard of a mariner equally noted for his kindliness                                      │
│  of heart and the respect and obedience paid to him by his crew, I felt                                         │
│  myself peculiarly fortunate in being able to secure his services. I heard                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  “What do you mean? What do you demand of your captain? Are you, then,                                          │
│  so easily turned from your design? Did you not call th

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Literary Researcher                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The captain of the ship that rescues Victor is the master of the vessel, who is described by Robert Walton as  │
│  "a person of an excellent disposition and is remarkable in the ship for his gentleness and the mildness of     │
│  his discipline" (Letter 1). This passage directly quotes the text, providing evidence of the captain's kind    │
│  nature.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Verifier                                                                                           │
│                                                                                                                 │
│  Task: Question: Who is the captain of the ship that rescues Victor?                                            │
│                                                                                                                 │
│  Passages from Frankenstein:                                                                                    │
│  In the morning, however, as soon as it was light, I went upon deck and                                         │
│  found all the sailors busy on one side of the vessel, apparently                                               │
│  talking to someone in the sea. It was, in fact, a sledge, like that we                                         │
│  had seen before, which had drifted towards us in the night on a large                                          │
│  fragment of ice. Only one dog remained alive; but there was a human                                            │
│  being within it whom the sailors were persuading to enter the vessel.                                          │
│  He was not, as the other traveller seemed to be, a savage inhabitant of                                        │
│  some undiscovered island, but a European. When I appeared on deck the                                          │
│  master said, “Here is our captain, and he will not allow you to perish                                         │
│  on the open sea.”                                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  The master is a person of an excellent disposition and is remarkable in the                                    │
│  ship for his gentleness and the mildness of his discipline. This                                               │
│  circumstance, added to his well-known integrity and dauntless courage, made                                    │
│  me very desirous to engage him. A youth passed in solitude, my best years                                      │
│  spent under your gentle and feminine fosterage, has so refined the                                             │
│  groundwork of my character that I cannot overcome an intense distaste to                                       │
│  the usual brutality exercised on board ship: I have never believed it to be                                    │
│  necessary, and when I heard of a mariner equally noted for his kindliness                                      │
│  of heart and the respect and obedience paid to him by his crew, I felt                                         │
│  myself peculiarly fortunate in being able to secure his services. I heard                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  “What do you mean? What do you demand of your captain? Are you, then,                                          │
│  so easily turned from your design? Did you not call th

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Verifier                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The master of the vessel, described by Robert Walton as "a person of an excellent disposition and is           │
│  remarkable in the ship for his gentleness and the mildness of his discipline," is the captain who rescues      │
│  Victor.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

A: The master of the vessel, described by Robert Walton as "a person of an excellent disposition and is remarkable in the ship for his gentleness and the mildness of his discipline," is the captain who rescues Victor.


💡 **Notice the extra time.** Agentic RAG is 3-5× slower than simple RAG because we're running multiple LLM calls in sequence. In exchange, we get a verification pass that often catches subtle hallucinations.

**When is it worth it?**
- ✅ High-stakes questions (medical, legal, financial)
- ✅ Multi-hop reasoning where one retrieval isn't enough
- ❌ Simple lookups or chat over small corpora

---

## Section 6 — Gradio chat UI

Wrap it all in a chat interface. Three lines of Gradio and we have a shareable app — `share=True` gives you a public URL that works for 1 week.

In [76]:
import gradio as gr

def chat_fn(message, history):
    """Generator — yields partial responses as tokens stream in."""
    partial = ""
    for chunk in simple_rag_chain.stream(message):
        partial += chunk
        yield partial

demo = gr.ChatInterface(
    fn=chat_fn,
    title="💀 Ask Frankenstein",
    description="A RAG-powered chatbot about Mary Shelley's 1818 novel.",
    examples=[
        "Does the monster have a name in Mary Shelley's Frankenstein?",
        "Who rescues Victor from the Arctic?",
        "What books does the creature read?",
        "Does the monster have a name?",
    ],
)

demo.launch(share=True, debug=False, inline=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4270ef458954b3a4d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


👆 Click the public link (ending in `.gradio.live`) to open the chat in a new tab. Try asking off-topic questions too — see if it sticks to the book.

---

## Section 7 — Head-to-Head: Plain LLM vs Simple RAG vs Agentic RAG 🏁

This is the finale. We ask the same questions to all three systems and compare answers side-by-side.

**Watch for:**
- 🎯 Which system hallucinates on *"monster's name"*?
- 🚫 Which system refuses *"capital of Switzerland"* (correctly, since it's out of scope)?
- ⏱️ How much longer does Agentic RAG take?

In [87]:
# Three approaches, same interface
def plain_llm(q):
    return ask_plain_llm(q, model='llama3.2:3b')

def simple_rag(q):
    return simple_rag_chain.invoke(q)

def agentic_rag(q):
    return run_agentic_rag(q)

TEST_QUESTIONS = [
    "Does the monster have a name in Mary Shelley's Frankenstein?", # Classic hallucination trap
    "Who is the captain of the ship that rescues Victor?",  # Factual, should be easy
    "What is the capital of Switzerland?",                  # Out of scope — should refuse
    "What books does the creature find and read in the forest?", # Specific factual detail
]

In [88]:
import time
import pandas as pd
from IPython.display import display, HTML

answerer_agent.verbose = False
verifier_agent.verbose = False

def run_and_time(fn, q, max_chars=500):
    t0 = time.time()
    try:
        answer = fn(q)
    except Exception as e:
        answer = f"[Error: {e}]"
    elapsed = time.time() - t0
    return f"⏱️ {elapsed:.1f}s\n\n{str(answer)[:max_chars]}"

results = []
for q in TEST_QUESTIONS:
    print(f"Running: {q[:70]}...")
    results.append({
        "Question":    q,
        "Plain LLM":   run_and_time(plain_llm, q),
        "Simple RAG":  run_and_time(simple_rag, q),
        "Agentic RAG": run_and_time(agentic_rag, q),
    })
    print("  ✓ done")

df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Render as a wide HTML table for clarity
styled = df.style.set_properties(**{
    'text-align': 'left',
    'vertical-align': 'top',
    'white-space': 'pre-wrap',
    'font-size': '12px',
    'padding': '8px',
    'border': '1px solid #ddd',
}).set_table_styles([
    {'selector': 'th', 'props': [('background', '#f0f0f0'), ('font-weight', 'bold'), ('text-align', 'left')]}
])
display(styled)

Running: Does the monster have a name in Mary Shelley's Frankenstein?...
  ✓ done
Running: Who is the captain of the ship that rescues Victor?...
  ✓ done
Running: What is the capital of Switzerland?...
  ✓ done
Running: What books does the creature find and read in the forest?...
  ✓ done


,Question,Plain LLM,Simple RAG,Agentic RAG
0,Does the monster have a name in Mary Shelley's Frankenstein?,"⏱️ 1.5s In Mary Shelley's novel ""Frankenstein,"" the creature is often referred to as ""the Monster"" or ""Adam."" However, it does not have a specific name given by the author. The creature is sometimes called ""Victor Frankenstein's Monster"" or simply ""the Creature,"" but these names are not officially assigned to it by the author. The creature is more of an unnamed entity, and its lack of a name serves to emphasize its isolation and otherness. It's worth noting that Mary Shelley does refer to the creature",⏱️ 0.4s The text does not specify.,"⏱️ 2.5s The monster refers to itself as ""the demoniacal corpse"" and addresses its creator as ""my tyrant and tormentor,"" indicating a sense of self-awareness and identity without a formal name."
1,Who is the captain of the ship that rescues Victor?,⏱️ 0.5s I don't have any information about a character named Victor or a ship that rescues him. Could you provide more context or clarify which story or series you are referring to? I'll do my best to help.,"⏱️ 0.6s The master, or captain, of the ship that rescues Victor is a person of an excellent disposition and is remarkable in the ship for his gentleness and mildness of discipline.","⏱️ 3.0s The master of the vessel, described as ""a person of an excellent disposition and is remarkable in the ship for his gentleness and the mildness of his discipline"" and noted for his integrity and dauntless courage, is the captain who rescues Victor. When Walton appears on deck, the master says, ""Here is our captain, and he will not allow you to perish on the open sea""."
2,What is the capital of Switzerland?,⏱️ 0.3s The capital of Switzerland is Bern.,⏱️ 0.4s The text does not specify.,⏱️ 1.8s The capital of Switzerland is not specified in the provided passages.
3,What books does the creature find and read in the forest?,⏱️ 0.4s I don't have any information about a specific creature finding and reading books in a forest. Could you provide more context or clarify which creature you are referring to? I'll do my best to help.,"⏱️ 0.7s The creature finds a leathern portmanteau containing several articles of dress and some books, including _Paradise Lost_, a volume of _Plutarch's Lives_, and the _Sorrows of Werter_.","⏱️ 4.2s The creature finds and reads in the forest the books ""Paradise Lost"", a volume of ""Plutarch's Lives"", and the ""Sorrows of Werter"". One night during my accustomed visit to the neighbouring wood where I collected my own food and brought home firing for my protectors, I found on the ground a leathern portmanteau containing several articles of dress and some books. I eagerly seized the prize and returned with it to my hovel. Fortunately the books were written in the language, the elements of which I"


### What typically happens

| Question | Plain LLM | Simple RAG | Agentic RAG |
|---|---|---|---|
| **Monster's name** | Often wrong ("Frankenstein") | "The text does not specify" | Same + verified |
| **Ship captain** | Sometimes right (Walton), sometimes hallucinates | Correct + grounded in letters | Correct + cross-checked |
| **Capital of Switzerland** | Answers "Bern" — **out of scope!** | Refuses (good!) | Refuses (good!) |
| **Books the creature reads** | Vague, may invent titles | Specific: *Paradise Lost*, *Plutarch's Lives*, *Sorrows of Werter* | Same + verified |

### Key takeaways

- **Plain LLM** — fastest, but dangerous: confident hallucinations + can't stay in scope
- **Simple RAG** — grounded, refuses to speculate, ~2-5× slower than plain
- **Agentic RAG** — most robust, catches errors the single-pass misses, 3-5× slower still

**No single approach is "best".** Pick based on:
- Accuracy requirements
- Latency budget
- Cost per query (more agents = more LLM calls)
- Question complexity (multi-hop → agentic wins; simple lookup → simple RAG is fine)

---

## Section 8 — Evaluation with Ragas (*optional / bonus*)

Eye-balling answers works for ~5 questions. For anything real you need **automatic metrics**. [Ragas](https://docs.ragas.io) is the standard open-source library — and, crucially, it runs against a local LLM judge, so we don't need an OpenAI key.

### Four core RAG metrics

| Metric | What it measures | What it needs |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? | answer + context |
| **Answer Relevancy** | Does the answer actually address the question? | question + answer + embeddings |
| **Context Precision** | Are the retrieved chunks actually useful (not noise)? | question + context + reference |
| **Context Recall** | Did retrieval surface everything needed to answer? | question + context + reference |

The first two grade the **generator**; the last two grade the **retriever**. Together they tell you *where* your pipeline is weakest.

> ⏱️ **Time budget:** ~2-3 min on GPU for 3 questions using `qwen3.5:9b` as the judge (plus a one-time ~1-2 min model pull). Skip this section if the clock is running out — the rest of the workshop stands on its own.

> 🤖 **Why a bigger judge than the pipeline?** Ragas metrics ask the judge model to return **structured JSON** (decomposed claims, entailment labels, generated sub-questions). `llama3.2:3b` — fine for generation — fails at this: it hallucinates Python code or writes prose when asked for JSON, producing `NaN` scores. We pull `qwen3.5:9b` specifically for its strong structured-output behavior and use it *only* as the judge. The Simple RAG / Agentic RAG pipelines still run on `llama3.2:3b`.

In [92]:
# Ragas isn't installed in Section 0.
# ~30 seconds to install.
%pip install -q ragas

# Pull a dedicated judge model. llama3.2:3b can't reliably emit the structured JSON Ragas metrics need.
# qwen3.5:9b is one of the strongest small judges for structured output — ~6GB,
# fits comfortably on a Colab T4 (16GB) alongside llama3.2:3b.

print("Pulling qwen3.5:9b as the judge (~6GB, ~1-2 min)...")
!ollama pull qwen3.5:9b

import os
os.environ["RAGAS_DO_NOT_TRACK"] = "true"

from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


judge_llm = LangchainLLMWrapper(
    ChatOllama(model="qwen3.5:9b", temperature=0, format="json")
)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)  # nomic-embed-text from Section 3

print("✅ Ragas ready (judge: qwen3.5:9b, embeddings: nomic-embed-text)")

Pulling qwen3.5:9b as the judge (~6GB, ~1-2 min)...

✅ Ragas ready (judge: qwen3.5:9b, embeddings: nomic-embed-text)


/tmp/ipykernel_7587/2248419139.py:24: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(
/tmp/ipykernel_7587/2248419139.py:27: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(embeddings)  # nomic-embed-text from Section 3


In [93]:
# A tiny curated eval set. Each entry has a question and the *reference*
# answer we'd expect from a careful reading of the book. In production these
# come from domain experts or LLM-generated + human-reviewed.
EVAL = [
    {
        "question": "Does the monster have a name in the novel?",
        "reference": (
            "No. The creature is never given a proper personal name. "
            "It is referred to as 'the creature', 'the wretch', 'the monster', "
            "'the demon', and 'the fiend', but never by a name."
        ),
    },
    {
        "question": "What books does the creature find and read in the forest?",
        "reference": (
            "Paradise Lost, a volume of Plutarch's Lives, and The Sorrows of Werter."
        ),
    },
    {
        "question": "Who writes the opening letters of the novel, and to whom?",
        "reference": (
            "Robert Walton, an English explorer on a voyage toward the North Pole, "
            "writing to his sister Mrs. Margaret Saville in England."
        ),
    },
]

# Run Simple RAG against each question and capture BOTH the answer and the
# retrieved chunks — Ragas scores retrieval too, not just the final answer.
from ragas import EvaluationDataset

samples = []
for ex in EVAL:
    q = ex["question"]
    docs = retriever.invoke(q)
    samples.append({
        "user_input":         q,
        "retrieved_contexts": [d.page_content for d in docs],
        "response":           simple_rag_chain.invoke(q),
        "reference":          ex["reference"],
    })

eval_dataset = EvaluationDataset.from_list(samples)
print(f"✅ Built eval dataset with {len(eval_dataset)} samples\n")
eval_dataset.to_pandas()[["user_input", "response"]]

✅ Built eval dataset with 3 samples



,user_input,response
0,Does the monster have a name in the novel?,The text does not specify.
1,What books does the creature find and read in the forest?,"The creature finds a leathern portmanteau containing several articles of dress and some books, including _Paradise Lost_, a volume of _Plutarch's Lives_, and the _Sorrows of Werter_."
2,"Who writes the opening letters of the novel, and to whom?",The text does not specify.


In [ ]:
# Four metrics — two grade the answer, two grade the retrieval.
# This runs ~12 LLM judge calls total (~2-3 min on GPU, ~8-10 min on CPU).

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas.metrics")

from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

result = evaluate(
    dataset=eval_dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        LLMContextPrecisionWithReference(),
        LLMContextRecall(),
    ],
    llm=judge_llm,
    embeddings=judge_embeddings,
)

scores_df = result.to_pandas()

# Score columns are the float-valued ones.
metric_cols = [c for c in scores_df.columns if scores_df[c].dtype.kind == "f"]

from IPython.display import display
print("Per-question scores:")
display(scores_df[["user_input"] + metric_cols].style.format(precision=2, na_rep="—"))

print("\nMean scores across all examples:")
print(scores_df[metric_cols].mean().round(3).to_string())

### Reading the scores

All four metrics live in **[0, 1]** — higher is better.

- **Faithfulness ≈ 1.0** → the answer doesn't invent facts beyond the retrieved chunks. Our Simple RAG prompt (*"use ONLY the context"*) is pulling its weight here.
- **Answer Relevancy ≈ 0.7–0.9** → answers address the question directly. Genuine `"The text does not specify."` replies will legitimately score lower — that's correct behavior, not a bug.
- **Context Precision** → fraction of retrieved chunks that actually support the reference answer. Low values mean `k` is too high, or the chunker is noisy.
- **Context Recall** → fraction of the reference answer that's *covered* by retrieved chunks. Low values mean the retriever is missing key passages — fix with better embeddings, hybrid (dense + BM25) search, or a bigger `k`.

### What to do with these numbers

- **Compare pipelines.** Rerun the two cells above but build `samples` from `run_agentic_rag(q)` instead of `simple_rag_chain.invoke(q)` — easiest way to see whether the verifier is paying its 2-3× latency cost.
- **Tune chunking.** Try `chunk_size=400` vs `chunk_size=1200` back in Section 2 and watch Context Precision/Recall move in opposite directions.
- **Tune `k`.** `k=3` vs `k=10` vs `k=20` — classic precision/recall trade-off.

> ⚠️ **Statistical caveat.** Three examples is fine for a smoke test, meaningless for real decisions. Aim for **≥ 50 questions** with human-reviewed references before you trust rank-ordered comparisons.

---

## Section 9 — Takeaways & Next Steps

### What you built today

| # | Component | Lines of code |
|---|---|---|
| 1 | Local LLM chat (Ollama) | ~5 |
| 2 | Chunking pipeline | ~10 |
| 3 | Embedding + vector store | ~8 |
| 4 | Simple RAG chain | ~15 |
| 5 | Agentic RAG (CrewAI) | ~40 |
| 6 | Gradio UI | ~10 |
| 7 | Comparison harness | ~20 |

**Under 120 lines for a working agentic RAG system.** That's the power of modern tooling.

### Where to go next

- 📄 **Swap the document**: replace `frankenstein.txt` with your research PDF, a product manual, your lab's knowledge base
- 🧠 **More agents**: add a Query Decomposer (splits multi-hop questions), a Ranker (reorders chunks), a Summarizer
- 🗄️ **Production vector DB**: ChromaDB → Pinecone, Weaviate, Qdrant, or Postgres + pgvector
- 🔍 **Hybrid search**: combine dense (embeddings) with sparse (BM25) for keyword-heavy corpora
- 📊 **Reranking**: use `bge-reranker` to score query-chunk pairs more precisely after retrieval
- 📡 **Production observability**: LangSmith or Langfuse for tracing + evaluation
- 🛡️ **Guardrails**: structured output, PII scrubbing, jailbreak detection (NeMo Guardrails, Guardrails AI)

### Resources

- Workshop page — https://hakanotal.github.io/workshop-rag-chatbot
- Source repo — https://github.com/hakanotal/workshop-rag-chatbot
- LangChain docs — https://python.langchain.com
- CrewAI docs — https://docs.crewai.com
- Ragas paper — https://arxiv.org/abs/2309.15217
- Ollama model library — https://ollama.com/library

### Questions?

**Thanks for coming!** 🙏